[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA1/IL1.1/2-langchain_model_api.ipynb)


# 2. LangChain Model API - Abstracción y Framework

## Objetivos de Aprendizaje
- Comprender las ventajas del framework LangChain sobre APIs directas
- Configurar `ChatOpenAI` para apuntar a distintos proveedores compatibles con OpenAI
- Explorar la compatibilidad entre diferentes modelos
- Implementar patrones de uso común con LangChain

## Introducción a LangChain

LangChain es un framework que simplifica el desarrollo de aplicaciones con modelos de lenguaje. Principales ventajas:
- **Abstracción**: Una interfaz unificada para múltiples proveedores
- **Herramientas**: Componentes predefinidos para tareas comunes
- **Cadenas**: Composición de múltiples operaciones
- **Memoria**: Gestión automática del historial de conversaciones

## Instalación de Dependencias
```bash
pip install langchain langchain-openai
```

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai openai python-dotenv


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "llama-3.3-70b-versatile")
    os.environ.setdefault("LLM_MODEL_SMALL", "llama-3.1-8b-instant")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

# Importar las bibliotecas de LangChain
from langchain_openai import ChatOpenAI
# LangChain v1: langchain_core.messages / langchain_core.documents
from langchain_classic.schema import HumanMessage, AIMessage, SystemMessage
import os

# Verificar versiones
print("Verificando instalación de LangChain...")
try:
    import langchain
    print(f"✓ LangChain version: {langchain.__version__}")
except ImportError:
    print("✗ LangChain no está instalado")

print("Bibliotecas importadas correctamente")

Verificando instalación de LangChain...
✓ LangChain version: 1.3.15
Bibliotecas importadas correctamente


In [3]:
# Configuración del modelo LangChain apuntando a Groq
try:
    llm = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
        temperature=0.7,
        max_tokens=150
    )
    
    print("✓ Modelo LangChain configurado correctamente")
    print(f"Modelo: {llm.model_name}")
    print(f"Temperature: {llm.temperature}")
    print(f"Max tokens: {llm.max_tokens}")
    
except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno OPENAI_BASE_URL y LLM_API_KEY")

✓ Modelo LangChain configurado correctamente
Modelo: llama-3.3-70b-versatile
Temperature: 0.7
Max tokens: 150


In [4]:
# Uso básico con LangChain - Diferentes tipos de mensajes
def ejemplo_basico():
    try:
        # Usar HumanMessage (equivalente a "user" en OpenAI)
        response = llm.invoke([HumanMessage(content="Hola, ¿cómo estás?")])
        print("=== Respuesta Básica ===")
        print(response.content)
        print(f"Tipo de respuesta: {type(response)}")
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejemplo básico
ejemplo_basico()

=== Respuesta Básica ===
**Hola, estoy bien, gracias**

Me alegra que hayas iniciado la conversación. Estoy aquí para ayudarte y responder a cualquier pregunta que tengas. ¿En qué puedo ayudarte hoy? ¿Necesitas información sobre algún tema en particular o simplemente quieres charlar un rato?
Tipo de respuesta: <class 'langchain_core.messages.ai.AIMessage'>


## Configuración Avanzada con LangChain

LangChain permite configuraciones más sofisticadas y cambiar proveedores fácilmente.

In [5]:
# Configuraciones múltiples con diferentes parámetros
def configuraciones_multiples():
    # Configuración conservadora (para tareas que requieren precisión)
    llm_conservador = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
        temperature=0.1,  # Muy determinístico
        max_tokens=100
    )
    
    # Configuración creativa (para tareas que requieren creatividad)
    llm_creativo = ChatOpenAI(
        base_url=os.getenv("LLM_BASE_URL"),
        api_key=os.getenv("LLM_API_KEY"),
        model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
        temperature=0.9,  # Muy creativo
        max_tokens=150
    )
    
    prompt = "Escribe un eslogan para una empresa de tecnología"
    
    print("=== COMPARACIÓN DE CONFIGURACIONES ===")
    
    try:
        # Respuesta conservadora
        print("\n1. Configuración Conservadora (temp=0.1):")
        print("-" * 40)
        response_conservador = llm_conservador.invoke([HumanMessage(content=prompt)])
        print(response_conservador.content)
        
        # Respuesta creativa
        print("\n2. Configuración Creativa (temp=0.9):")
        print("-" * 35)
        response_creativo = llm_creativo.invoke([HumanMessage(content=prompt)])
        print(response_creativo.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar comparación
configuraciones_multiples()

=== COMPARACIÓN DE CONFIGURACIONES ===

1. Configuración Conservadora (temp=0.1):
----------------------------------------


"Conecta con el futuro, innova con nosotros"

2. Configuración Creativa (temp=0.9):
-----------------------------------


"Conecta con el futuro, innova con nosotros"


## Comparación: LangChain vs cliente `openai` directo

Veamos las diferencias en código entre usar LangChain y el cliente `openai` directo. Ojo: en ambos casos el proveedor es el mismo (Groq); lo que cambia es la capa de abstracción con la que lo llamamos.

In [6]:
# Comparación de código entre LangChain y el cliente `openai` directo
from openai import OpenAI

def comparar_enfoques():
    prompt = "Explica qué es Python en una oración"
    
    print("=" * 60)
    print("COMPARACIÓN: LangChain vs cliente openai directo")
    print("=" * 60)
    
    # Método 1: cliente `openai` directo
    print("\n1. Cliente openai directo:")
    print("-" * 20)
    try:
        client = OpenAI(
            base_url=os.getenv("LLM_BASE_URL"),
            api_key=os.getenv("LLM_API_KEY")
        )
        
        response_openai = client.chat.completions.create(
            model=os.getenv("LLM_MODEL", "llama-3.3-70b-versatile"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=50
        )
        
        print(f"Respuesta: {response_openai.choices[0].message.content}")
        print(f"Tokens: {response_openai.usage.total_tokens}")
        
    except Exception as e:
        print(f"Error con el cliente openai: {e}")
    
    # Método 2: LangChain
    print("\n2. LangChain:")
    print("-" * 15)
    try:
        response_langchain = llm.invoke([HumanMessage(content=prompt)])
        print(f"Respuesta: {response_langchain.content}")
        print(f"Tipo: {type(response_langchain)}")
        
    except Exception as e:
        print(f"Error LangChain: {e}")
    
    print("\n" + "=" * 60)
    print("VENTAJAS DE CADA ENFOQUE:")
    print("=" * 60)
    print("Cliente openai directo:")
    print("+ Control total sobre parámetros")
    print("+ Acceso directo a metadatos (tokens, costos)")
    print("+ Menor abstracción, más transparente")
    print()
    print("LangChain:")
    print("+ Interfaz unificada para múltiples proveedores")
    print("+ Herramientas adicionales (cadenas, memoria, etc.)")
    print("+ Más fácil composición de operaciones complejas")
    print("+ Mejor para prototipado rápido")

# Ejecutar comparación
comparar_enfoques()

COMPARACIÓN: LangChain vs cliente openai directo

1. Cliente openai directo:
--------------------


Respuesta: Python es un lenguaje de programación de alto nivel, interpretado y orientado a objetos, ampliamente utilizado para desarrollar aplicaciones de todo tipo, desde scripts sencillos hasta sistemas complejos, gracias a su sintaxis cl
Tokens: 94

2. LangChain:
---------------


Respuesta: Python es un lenguaje de programación de alto nivel, interpretado y orientado a objetos, ampliamente utilizado para el desarrollo de aplicaciones, análisis de datos, inteligencia artificial y más, conocido por su sintaxis simple y legible.
Tipo: <class 'langchain_core.messages.ai.AIMessage'>

VENTAJAS DE CADA ENFOQUE:
Cliente openai directo:
+ Control total sobre parámetros
+ Acceso directo a metadatos (tokens, costos)
+ Menor abstracción, más transparente

LangChain:
+ Interfaz unificada para múltiples proveedores
+ Herramientas adicionales (cadenas, memoria, etc.)
+ Más fácil composición de operaciones complejas
+ Mejor para prototipado rápido


## Tipos de Mensajes en LangChain

LangChain proporciona diferentes tipos de mensajes que corresponden a los roles en OpenAI:
- **HumanMessage**: Mensajes del usuario (equivale a "user")
- **AIMessage**: Respuestas del asistente (equivale a "assistant")  
- **SystemMessage**: Instrucciones del sistema (equivale a "system")

In [7]:
# Ejemplo con múltiples tipos de mensajes
def ejemplo_conversacion_completa():
    try:
        messages = [
            SystemMessage(content="Eres un tutor de programación amigable y paciente. Explicas conceptos técnicos de forma clara y das ejemplos prácticos."),
            HumanMessage(content="¿Qué es una función en programación?"),
            AIMessage(content="Una función es un bloque de código reutilizable que realiza una tarea específica. Te ayuda a organizar tu código y evitar repetición."),
            HumanMessage(content="¿Puedes darme un ejemplo simple en Python?")
        ]
        
        response = llm.invoke(messages)
        print("=== Conversación con Contexto ===")
        print(response.content)
        
    except Exception as e:
        print(f"Error: {e}")

# Ejecutar ejemplo
ejemplo_conversacion_completa()

=== Conversación con Contexto ===
**Ejemplo de una función simple en Python**

```python
# Definimos una función que saluda al usuario
def saludar(nombre):
    print(f"Hola, {nombre}!")

# Llamamos a la función con un nombre
saludar("Juan")
```

En este ejemplo, la función `saludar` toma un parámetro `nombre` y imprime un mensaje de saludo personalizado. Luego, llamamos a la función con el nombre "Juan" como argumento. El resultado será:

```
Hola, Juan!
```

**Ventajas de usar funciones**

*   Reutilización de código: Puedes llamar a la función varias veces sin tener que repet


## Ejercicios Prácticos

### Ejercicio 1: Crear Diferentes Personalidades
Usa SystemMessage para crear asistentes con diferentes personalidades (formal, casual, técnico, creativo).

### Ejercicio 2: Cadena de Conversación
Construye una conversación de múltiples turnos usando los diferentes tipos de mensajes.

### Ejercicio 3: Comparar Proveedores
Si tienes acceso a múltiples proveedores, configura LangChain para usar diferentes APIs y compara resultados.

## Conceptos Clave Aprendidos

1. **Abstracción de LangChain** sobre APIs directas
2. **Tipos de mensajes** y su equivalencia con roles OpenAI
3. **Configuraciones múltiples** para diferentes casos de uso
4. **Ventajas y desventajas** de frameworks vs APIs directas
5. **Intercambiabilidad** de proveedores con LangChain

## Próximos Pasos

En el siguiente notebook exploraremos el **streaming** con LangChain, que permite mostrar respuestas en tiempo real para mejorar la experiencia de usuario.